# Jawad Hassan
# 2230-0035
# BS AI
# ANN
# Lab 09


### Task:
### RNN WITH SPAM DATASET

In [2]:
from google.colab import files
uploaded = files.upload()

!pip install kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d uciml/sms-spam-collection-dataset
!unzip sms-spam-collection-dataset.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
License(s): unknown
100% 211k/211k [00:00<00:00, 140MB/s]

Archive:  sms-spam-collection-dataset.zip
  inflating: spam.csv                


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# 1. Load the dataset
# The SMS Spam dataset typically requires 'latin-1' encoding.
# It also often contains extra empty columns that need to be dropped.
df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']] # Keep only the label (v1) and message (v2) columns
df.columns = ['label', 'message'] # Rename for clarity

# 2. Preprocess labels and text
X = df['message'].values
# Convert 'spam' to 1 and 'ham' (not spam) to 0
y = np.where(df['label'] == 'spam', 1, 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# We can reduce the vocabulary size since SMS language is more limited
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# SMS messages are short; a maxlen of 50 covers almost all text messages
maxlen = 50
X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen)
X_test_pad = pad_sequences(X_test_seq, maxlen=maxlen)

# 3. Build the Stacked SimpleRNN Model
model = Sequential()
# Output dim reduced slightly as the vocabulary and complexity are lower
model.add(Embedding(input_dim=5000, output_dim=64, input_length=maxlen))
model.add(SimpleRNN(32, return_sequences=True, dropout=0.2, recurrent_dropout=0.2))
model.add(SimpleRNN(16, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# 4. Compile and Train
optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Batch size reduced to 32, which often generalizes better on smaller datasets
model.fit(X_train_pad, y_train, epochs=20, batch_size=32, validation_split=0.2, callbacks=[early_stop])

# 5. Evaluate
loss, accuracy = model.evaluate(X_test_pad, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 16s 68ms/step - accuracy: 0.7492 - loss: 0.5591 - val_accuracy: 0.8621 - val_loss: 0.3996
Epoch 2/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8662 - loss: 0.4217 - val_accuracy: 0.8621 - val_loss: 0.3929
Epoch 3/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9285 - loss: 0.2666 - val_accuracy: 0.9798 - val_loss: 0.0833
Epoch 4/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9832 - loss: 0.0970 - val_accuracy: 0.9877 - val_loss: 0.0593
Epoch 5/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9888 - loss: 0.0614 - val_accuracy: 0.9843 - val_loss: 0.0763
Epoch 6/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9927 - loss: 0.0397 - val_accuracy: 0.9821 - val_loss: 0.1001
Epoch 7/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9947 - loss: 0.0346 - val_accuracy: 0.9832 - val_loss: 0.1033
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9821 - loss: 0.0830
Test Accuracy: 0.9821


In [4]:
def predict_spam(message_text):
    # 1. Convert text to sequence using the tokenizer fitted on the SMS data
    seq = tokenizer.texts_to_sequences([message_text])

    # 2. Pad the sequence to the new maxlen of 50
    pad = pad_sequences(seq, maxlen=50)

    # 3. Predict (returns a probability between 0 and 1)
    prediction = model.predict(pad, verbose=0)[0][0]

    # 4. Classify based on a 0.5 threshold
    if prediction >= 0.5:
        return "Spam", prediction
    else:
        return "Not Spam", prediction

# --- Let's test it out ---

# Example 1: Classic Spam
spam_msg = "Free entry in 2 a wkly comp! Text WIN to 87077 to claim your prize."
result1, score1 = predict_spam(spam_msg)
print(f"Message: '{spam_msg}'")
print(f"Prediction: {result1} (Probability: {score1:.4f})\n")

# Example 2: Normal Text (Ham)
normal_msg = "Hey Jawad, are we still meeting up in Rawalpindi later today?"
result2, score2 = predict_spam(normal_msg)
print(f"Message: '{normal_msg}'")
print(f"Prediction: {result2} (Probability: {score2:.4f})")

Message: 'Free entry in 2 a wkly comp! Text WIN to 87077 to claim your prize.'
Prediction: Spam (Probability: 0.9784)

Message: 'Hey Jawad, are we still meeting up in Rawalpindi later today?'
Prediction: Not Spam (Probability: 0.0034)
